In [ ]:
import pandas as pd
import numpy as np
import holidays

In [ ]:
class HolidayFeatures:
    """Calculates holidays and term times"""
    
    @staticmethod
    def get_uk_holidays():
        """Generates UK holidays, extra bank holidays, and university closure periods."""
        uk_holidays = holidays.CountryHoliday('UK', subdiv='England', years=[2021, 2022, 2023])
        holiday_dates = list(uk_holidays.keys())
        
        extra_bank_holidays = [
            pd.to_datetime('2022-06-02').date(), pd.to_datetime('2022-06-03').date(), 
            pd.to_datetime('2022-09-19').date(), pd.to_datetime('2023-05-08').date()  
        ]
        holiday_dates.extend(extra_bank_holidays)

        closure_periods = pd.concat([
            pd.Series(pd.date_range(start='2022-12-23', end='2023-01-03')),
            pd.Series(pd.date_range(start='2023-12-23', end='2024-01-02'))
        ]).dt.date.tolist()
        
        holiday_dates.extend(closure_periods)
        return set(holiday_dates)

    @staticmethod
    def is_term_time(date_obj):
        """Determines if a date falls within university term times."""
        m = date_obj.month
        d = date_obj.day
        if (m == 6 and d > 15) or m == 7 or m == 8 or (m == 9 and d < 25): return 0
        if (m == 3 and d > 25) or (m == 4 and d < 20): return 0
        if (m == 12 and d > 15) or (m == 1 and d < 15): return 0
        return 1

In [ ]:
class ConFeatures:
    @classmethod
    def create_consumption_features(cls, deop, solcast, steps_per_day=288, days_ahead=1):
        """Builds all features required for the consumption model."""
        features = pd.DataFrame(index=deop.index)
        
        # Weather Features
        features['air_temp'] = solcast['air_temp']
        features['heating_demand'] = np.maximum(0, 18 - features['air_temp'])
        features['cooling_demand'] = np.maximum(0, features['air_temp'] - 18)
        
        # Time Variables
        features['hour'] = features.index.hour
        features['day_of_week'] = features.index.dayofweek
        features['month'] = features.index.month
        features['is_weekend'] = features.index.dayofweek.isin([5, 6]).astype(int)
        
        # Holiday / Term Time
        holiday_set = HolidayFeatures.get_uk_holidays()
        features['is_holiday'] = features.index.date
        features['is_holiday'] = features['is_holiday'].apply(lambda d: 1 if d in holiday_set else 0)
        features['non_working_day'] = np.maximum(features['is_weekend'], features['is_holiday'])
        features['non_working_hour_interaction'] = features['non_working_day'] * features['hour']
        
        features['is_term_time'] = features.index.date
        features['is_term_time'] = features['is_term_time'].apply(HolidayFeatures.is_term_time)
        
        # Cyclical Time 
        time_float = features.index.hour + features.index.minute / 60.0
        features['hour_sin'] = np.sin(2 * np.pi * time_float / 24)
        features['hour_cos'] = np.cos(2 * np.pi * time_float / 24)
        features['day_sin'] = np.sin(2 * np.pi * features.index.dayofweek / 7)
        features['day_cos'] = np.cos(2 * np.pi * features.index.dayofweek / 7)

        # Smart Lagging
        base_shift = steps_per_day * days_ahead
        features['load_lag_1d'] = deop['power-con-ave'].shift(base_shift)
        
        shift_7d = max(7, days_ahead) * steps_per_day
        features['load_lag_7d'] = deop['power-con-ave'].shift(shift_7d)
        
        yesterday_non_working = features['non_working_day'].shift(base_shift).fillna(0)
        features['smart_lag'] = np.where(
            (features['non_working_day'] == 0) & (yesterday_non_working == 0), 
            features['load_lag_1d'], 
            features['load_lag_7d']
        )
        
        # Rolling Baselines
        daily_mean = deop['power-con-ave'].resample('D').mean()
        daily_max = deop['power-con-ave'].resample('D').max()
        
        features['yesterday_daily_mean'] = features.index.normalize().map(daily_mean.shift(days_ahead))
        features['yesterday_daily_max'] = features.index.normalize().map(daily_max.shift(days_ahead))
        
        rolling_7d_mean = daily_mean.rolling(7).mean()
        features['rolling_7d_baseline'] = features.index.normalize().map(rolling_7d_mean.shift(days_ahead))

        return features.dropna()

In [ ]:
class PVFeatures:
    @classmethod
    def create_pv_features(cls, deop, solcast, days_ahead=1, steps_per_day=288):
        """Builds all features required for the PV Generation model."""
        df = pd.DataFrame(index=deop.index)
        
        # Weather
        df['gti_target'] = solcast['gti']
        df['cloud_opacity_target'] = solcast['cloud_opacity']
        df['snow_depth'] = solcast['snow_depth']
        
        # Shift for days ahead
        base_shift = steps_per_day * days_ahead
        
        # Power Lags
        df['lag_1d'] = deop['power-gen-pv-ave'].shift(base_shift)
        df['lag_2d'] = deop['power-gen-pv-ave'].shift(base_shift + steps_per_day)
        df['lag_3d'] = deop['power-gen-pv-ave'].shift(base_shift + steps_per_day * 2)
        
        shift_7d = max(7, days_ahead) * steps_per_day
        shift_14d = max(14, days_ahead) * steps_per_day
        df['lag_7d'] = deop['power-gen-pv-ave'].shift(shift_7d)
        df['lag_14d'] = deop['power-gen-pv-ave'].shift(shift_14d)

        # Volatility & Trend 
        df['power_volatility_1d'] = deop['power-gen-pv-ave'].shift(base_shift).rolling(24).std()
        df['power_trend_1d'] = df['lag_1d'] - df['lag_2d']
        df['gti_volatility_1d'] = solcast['gti'].shift(base_shift).rolling(24).std()
        
        # Peak of previous day 
        daily_max = deop['power-gen-pv-ave'].resample('D').max().shift(days_ahead)
        df['yesterday_peak'] = df.index.normalize().map(daily_max)

        # Geometry & Time
        hour_num = df.index.hour + df.index.minute / 60
        df['solar_potential'] = np.maximum(0, np.sin(np.pi * (hour_num - 6) / 12))
        df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)

        return df.dropna()